# Telecom RAG Agent — Create & Test

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CrashingGuru/Sandbox_Training/blob/main/notebooks/telecom-rag-agent/notebook/Telecom_RAG_Agent.ipynb)

Companion notebook for the **Telecom RAG Agent** in the agent catalog. It loads the packaged agent definition (`agent.py`), wires it to a retriever and a small LLM, and runs test queries — showing the agent's **tool trace** (`retrieve → answer` or `retrieve → refuse`).

The agent's logic lives in a reusable module; this notebook is just how you **create and test** it. Runs on the Colab free tier (CPU).

In [ ]:
!pip -q install pyzipper

import json, urllib.request, random
import pyzipper

ZIP_URL = "https://raw.githubusercontent.com/netop-team/TeleQnA/main/TeleQnA.zip"
urllib.request.urlretrieve(ZIP_URL, "TeleQnA.zip")
with pyzipper.AESZipFile("TeleQnA.zip") as z:
    z.setpassword(b"teleqnadataset")   # documented in the TeleQnA README
    data = json.loads(z.read("TeleQnA.txt"))

def options_of(v):
    return {int(k.split()[1]): v[k] for k in v if k.startswith('option ')}
def correct_idx(v):
    import re; m = re.search(r'option\s*(\d+)', v.get('answer', '')); return int(m.group(1)) if m else None

items = [v for v in data.values() if correct_idx(v) and options_of(v)]
random.Random(42).shuffle(items)
POOL, EVAL = items[:500], items[500:510]
kb_docs = [f"{v['question']} {v.get('explanation','')}".strip() for v in POOL]
print("KB docs:", len(kb_docs), "| test questions:", len(EVAL))

## 1. Load the packaged agent definition

Fetch `agent.py` straight from the catalog so the notebook always uses the published version.

In [ ]:
AGENT_URL = ("https://raw.githubusercontent.com/CrashingGuru/Sandbox_Training/main/"
             "agents/telecom-rag-agent/agent/agent.py")
urllib.request.urlretrieve(AGENT_URL, "agent.py")
from agent import TelecomRAGAgent
print("Loaded:", TelecomRAGAgent.__doc__.splitlines()[0])

## 2. Wire up the retriever and the LLM

The agent injects these dependencies, so you can swap in any embedder / LLM later.

In [ ]:
!pip -q install sentence-transformers transformers

from sentence_transformers import SentenceTransformer
from transformers import pipeline

embedder = SentenceTransformer("all-MiniLM-L6-v2")
llm = pipeline("text2text-generation", model="google/flan-t5-base")
print("retriever + LLM ready")

## 3. Create the agent and test it

Each test prints the agent's **tool trace**, so you can see it retrieve and then answer or refuse.

In [ ]:
agent = TelecomRAGAgent(embedder, llm, kb_docs, tau=0.25)

for v in EVAL[:5]:
    options = options_of(v)
    pred = agent.run(v["question"], options)
    verdict = "REFUSED" if pred is None else ("correct" if pred == correct_idx(v) else "wrong")
    print(f"Q: {v['question'][:80]}...")
    print(f"   trace: {agent.log}")
    print(f"   agent={pred}  gold={correct_idx(v)}  -> {verdict}\n")

## 4. Notes

- **Reuse:** `agent.py` is a standalone module — import it anywhere, inject your own embedder/LLM, and call `run(question, options)`.
- **Guardrail:** the agent **refuses** (returns `None`) when retrieval confidence is below `tau`, instead of guessing. This is the governance control point.
- **Auditability:** every tool call is recorded in `agent.log`.
- **Scale up:** point the retriever at real 3GPP/IEEE specifications and swap in a stronger LLM; the retrieve → answer/refuse contract stays the same.